# Irrigation_Need


In [2]:
import json
from datetime import datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import catboost as cb
import lightgbm as lgbm

Load Data

In [3]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

orig = pd.read_csv("irrigation_prediction.csv")

print(train_df.head())

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

Preprocess

In [4]:
TARGET_COL = "Irrigation_Need"
ID_COL = "id"

X = train_df.drop(columns=[ID_COL, TARGET_COL])
y = train_df[TARGET_COL].copy()
X_test = test_df.drop(columns=[ID_COL])

num_col = X.select_dtypes(include="number").columns.tolist()
cat_col = X.select_dtypes(exclude="number").columns.tolist()
print(f"len(num_col): {len(num_col)}")
print(f"num_col: {num_col}")
print(f"len(cat_col): {len(cat_col)}")
print(f"cat_col: {cat_col}")


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    基於領域知識和常見特徵工程技巧創造新特徵。
    """

    df = df.copy()

    # # base
    df["dryness_index"] = 100 - df["Soil_Moisture"]
    df["moisture_sq"] = df["Soil_Moisture"] ** 2
    df["temp_sq"] = df["Temperature_C"] ** 2
    df["moisture_rank"] = df["Soil_Moisture"].rank()

    # binary features
    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype(np.int8)
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype(np.int8)
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype(np.int8)
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype(np.int8)

    # new test
    df["magic_harvest"] = (df["Crop_Growth_Stage"] == "Harvest").astype(np.int8)
    df["magic_sowing"] = (df["Crop_Growth_Stage"] == "Sowing").astype(np.int8)
    df["magic_mulch_yes"] = (df["Mulching_Used"] == "Yes").astype(np.int8)
    df["magic_high_score"] = (df["soil_lt_25"] * 2 + df["rain_lt_300"] * 2+ df["temp_gt_30"] + df["wind_gt_10"])
    df["magic_low_score"] = (df["magic_harvest"] * 2 + df["magic_sowing"] * 2 + df["magic_mulch_yes"])
    df["magic_score"] = df["magic_high_score"] - df["magic_low_score"]
    df["magic_dist_to_boundary"] = np.minimum(np.abs(df["magic_score"] - 0), np.abs(df["magic_score"] - 3))
    df["margin_soil_25"] = df["Soil_Moisture"] - 25
    df["margin_rain_300"] = df["Rainfall_mm"] - 300
    df["margin_temp_30"] = df["Temperature_C"] - 30
    df["margin_wind_10"] = df["Wind_Speed_kmh"] - 10 
    
    df["north_region"] = (df['Region'] == 'North').astype(np.int8)    

    return df

X = add_features(X)
X_test = add_features(X_test)

len(num_col): 11
num_col: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
len(cat_col): 8
cat_col: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [5]:
def encode_fold_features(X_trn_raw: pd.DataFrame, X_test_raw: pd.DataFrame):
    """
    對每個類別特徵轉碼成xgboost可接受的格式。
    """

    X_trn = X_trn_raw.copy()
    X_tst = X_test_raw.copy()

    # Use native categorical support in XGBoost (no manual one-hot)
    cat_cols = X_trn.select_dtypes(include=["object", "category", "string"]).columns.tolist()

    for col in cat_cols:
        trn_s = X_trn[col].astype(str)
        tst_s = X_tst[col].astype(str)

        # lock categories to training fold to avoid leakage
        trn_cat = pd.Categorical(trn_s)
        categories = trn_cat.categories

        X_trn[col] = trn_cat
        X_tst[col] = pd.Categorical(tst_s, categories=categories)

    # drop no-signal columns in this fold
    const_cols = [c for c in X_trn.columns if X_trn[c].nunique(dropna=False) <= 1]
    if const_cols:
        X_trn = X_trn.drop(columns=const_cols)
        X_tst = X_tst.drop(columns=const_cols, errors="ignore")

    return X_trn, X_tst

X, X_test = encode_fold_features(X, X_test)

In [6]:
# import seaborn as sns
# for c in num_col:
#     plt.figure(figsize=(5,3))
#     sns.boxplot(data=X, x=y, y=c)
#     plt.title(f'{c} by y')
#     plt.show()

# for c in cat_col:
#     plt.figure(figsize=(6,3))
#     sns.countplot(data=X, x=c, hue=y)
#     plt.xticks(rotation=45)
#     plt.title(f'{c} vs y (count)')
#     plt.show()

In [7]:
LABEL2ID = {"Low": 0, "Medium": 1, "High": 2}
ID2LABEL = {0: "Low", 1: "Medium", 2: "High"}

y_encoded = y.map(LABEL2ID)
y_encoded = y_encoded.to_numpy(dtype=int)

orig_y_encoded = orig[TARGET_COL].map(LABEL2ID)
orig_y_encoded = orig_y_encoded.to_numpy(dtype=int)

num_classes = len(LABEL2ID)

print("label mapping:", LABEL2ID)
print("orig rows:", len(orig_y_encoded))


label mapping: {'Low': 0, 'Medium': 1, 'High': 2}
orig rows: 10000


In [8]:
def target_encode(train_df, test_df, y, cols, n_splits=5, alpha=10):
    """
    對類別特徵進行目標編碼，使用KFold避免數據泄漏。
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    global_mean = y.mean()

    for col in cols:
        new_col = f"{col}_te"
        train_df[new_col] = np.zeros(len(train_df), dtype=np.float32)

        for train_idx, val_idx in kf.split(train_df):
            X_tr, X_val = train_df.iloc[train_idx], train_df.iloc[val_idx]
            y_tr = y[train_idx]

            stats = pd.DataFrame({col: X_tr[col], "target": y_tr})
            stats = stats.groupby(col)["target"].agg(["mean", "count"])

            smooth = (stats["mean"] * stats["count"] + global_mean * alpha) / (stats["count"] + alpha)

            train_df.loc[val_idx, new_col] = (X_val[col].map(smooth).astype(np.float32))

        # test
        stats = pd.DataFrame({col: train_df[col], "target": y})
        stats = stats.groupby(col)["target"].agg(["mean", "count"])
        smooth = (stats["mean"] * stats["count"] + global_mean * alpha) / (stats["count"] + alpha)

        test_df[new_col] = test_df[col].map(smooth).astype(np.float32)
        test_df[new_col] = test_df[new_col].fillna(global_mean).astype(np.float32)

    return train_df, test_df

X, X_test = target_encode(X, X_test, y_encoded, X.columns.tolist())

In [9]:
def freq_encoding(train_df, test_df, cols):
    """
    對類別特徵進行頻率編碼。
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in cols:
        freq = train_df[col].astype(str).value_counts()

        # TRAIN
        train_df[f"{col}_freq"] = train_df[col].astype(str).map(freq).astype(np.float32)

        # TEST
        test_df[f"{col}_freq"] = (
            test_df[col]
            .astype(str)
            .map(freq)
            .fillna(0)
            .astype(np.float32)
        )

        train_df[f"{col}_freq_norm"] = train_df[f"{col}_freq"] / len(train_df)
        test_df[f"{col}_freq_norm"] = test_df[f"{col}_freq"] / len(train_df)

    return train_df, test_df

X, X_test = freq_encoding(X, X_test, cat_col)

In [10]:
print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y: {y_encoded.shape}")
# X.head(5).to_csv("X.csv", index=False)
# classes = np.unique(y)
# print(classes)
# class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
# print(class_weights)
# weight_map = dict(zip(classes, class_weights))
# print(weight_map)
# sample_weights = np.array([weight_map[t] for t in y], dtype=np.float32)
# print(sample_weights)

X shape: (630000, 94)
X_test shape: (270000, 94)
y: (630000,)


In [11]:
def balanced_accuracy():
    def xgb_balanced_accuracy(y_true, y_pred):
        n_classes = 3
        y_pred_labels = np.argmax(y_pred.reshape(-1, n_classes), axis=1)
        return balanced_accuracy_score(y_true.astype(int), y_pred_labels)

    xgb_balanced_accuracy.__name__ = 'bal_ACC'
    return xgb_balanced_accuracy

Model Fit

In [12]:
xgb_params = {
    "n_estimators": 50000,
    "objective": "multi:softprob",
    "learning_rate": 0.01,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "num_class": num_classes,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda",
    "enable_categorical": True,
    "max_cat_to_onehot": 1,
    "max_bin": 1024,
    "n_jobs": -1,
}

cat_params = {
    "task_type": "GPU",
    "iterations": 10000,
    "learning_rate": 0.03,
    "depth": 6,
    "auto_class_weights": "Balanced",
    "cat_features": cat_col,
    "verbose": 1000,
    "random_seed": 42,
    "eval_metric": "TotalF1:average=Macro"
}

lgbm_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "n_estimators": 3000,
    "learning_rate": 0.04,
    "num_leaves": 127,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "verbose": -1,
    "n_jobs": -1,
    "device": "gpu",
    "gpu_platform_id": 0,
    "gpu_device_id": -1,             # use all GPUs
    "gpu_use_dp": True, 
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof_pred = np.zeros(len(X), dtype=int)
xgb_oof_proba = np.zeros((len(X), num_classes), dtype=float)
xgb_test_proba_folds = np.zeros((len(X_test), num_classes), dtype=float)

cat_oof_pred = np.zeros(len(X), dtype=int)
cat_oof_proba = np.zeros((len(X), num_classes), dtype=float)
cat_test_proba_folds = np.zeros((len(X_test), num_classes), dtype=float)

lgbm_oof_pred = np.zeros(len(X), dtype=int)
lgbm_oof_proba = np.zeros((len(X), num_classes), dtype=float)
lgbm_test_proba_folds = np.zeros((len(X_test), num_classes), dtype=float)

xgb_fold_bal_acc = []
cat_fold_bal_acc = []
lgbm_fold_bal_acc = []

train_columns = None

gain_importances = []
perm_importances = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y_encoded), 1):
    X_trn = X.iloc[trn_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_testing = X_test.copy()

    y_trn = y_encoded[trn_idx]
    y_val = y_encoded[val_idx]

    if train_columns is None:
        train_columns = X_trn.columns.tolist()

    classes = np.unique(y_trn)
    class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_trn)
    weight_map = dict(zip(classes, class_weights))
    sample_weights = np.array([weight_map[t] for t in y_trn], dtype=np.float32)

    # XGBoost
    xgb_model = xgb.XGBClassifier(
        **xgb_params,
        eval_metric=balanced_accuracy(),
        callbacks=[xgb.callback.EarlyStopping(rounds=500, metric_name='bal_ACC', maximize=True, save_best=True)]
    )

    xgb_model.fit(
        X_trn,
        y_trn,
        eval_set=[(X_val, y_val)],
        sample_weight=sample_weights,
        verbose=1000,
    )

    xgb_val_proba = xgb_model.predict_proba(X_val)
    xgb_val_pred = np.argmax(xgb_val_proba, axis=1)

    xgb_oof_proba[val_idx] = xgb_val_proba
    xgb_oof_pred[val_idx] = xgb_val_pred

    xgb_bal_acc = balanced_accuracy_score(y_val, xgb_val_pred)
    xgb_fold_bal_acc.append(xgb_bal_acc)

    print(f"Fold {fold}: XGB_BAL_ACC={xgb_bal_acc:.5f}, n_features={X_trn.shape[1]}")

    xgb_test_proba_folds += xgb_model.predict_proba(X_testing) / skf.n_splits

    # CatBoost
    cat_model = cb.CatBoostClassifier(
        **cat_params,
    )

    cat_model.fit(
        X_trn,
        y_trn,
        eval_set=[(X_val, y_val)],
        verbose=1000,
        early_stopping_rounds=500,
    )

    cat_val_proba = cat_model.predict_proba(X_val)
    cat_val_pred = np.argmax(cat_val_proba, axis=1)

    cat_oof_proba[val_idx] = cat_val_proba
    cat_oof_pred[val_idx] = cat_val_pred

    cat_bal_acc = balanced_accuracy_score(y_val, cat_val_pred)
    cat_fold_bal_acc.append(cat_bal_acc)

    print(f"Fold {fold}: CAT_BAL_ACC={cat_bal_acc:.5f}, n_features={X_trn.shape[1]}")

    cat_test_proba_folds += cat_model.predict_proba(X_testing) / skf.n_splits

    # LightGBM
    lgbm_model = lgbm.LGBMClassifier(
        **lgbm_params,
    )

    lgbm_model.fit(
        X_trn,
        y_trn,
        eval_set=[(X_val, y_val)],
        sample_weight=sample_weights,
        callbacks=[
            lgbm.early_stopping(150, verbose=False),
            lgbm.log_evaluation(0)
        ],
    )

    lgbm_val_proba = lgbm_model.predict_proba(X_val)
    lgbm_val_pred = np.argmax(lgbm_val_proba, axis=1)

    lgbm_oof_proba[val_idx] = lgbm_val_proba
    lgbm_oof_pred[val_idx] = lgbm_val_pred

    lgbm_bal_acc = balanced_accuracy_score(y_val, lgbm_val_pred)
    lgbm_fold_bal_acc.append(lgbm_bal_acc)

    print(f"Fold {fold}: LGBM_BAL_ACC={lgbm_bal_acc:.5f}, n_features={X_trn.shape[1]}")

    lgbm_test_proba_folds += lgbm_model.predict_proba(X_testing) / skf.n_splits

    # gain = xgb_model.get_booster().get_score(importance_type="gain")
    # gain_series = pd.Series(gain).reindex(X_trn.columns, fill_value=0.0)
    # gain_importances.append(gain_series)

    # # GPU memory-safe permutation importance: use a subset + single process
    # perm_n = min(20000, len(X_val))
    # X_val_perm = X_val.iloc[:perm_n]
    # y_val_perm = y_val[:perm_n]
    # perm = permutation_importance(
    #     xgb_model,
    #     X_val_perm,
    #     y_val_perm,
    #     scoring="balanced_accuracy",
    #     n_repeats=2,
    #     random_state=42,
    #     n_jobs=1,
    # )
    # perm_series = pd.Series(perm.importances_mean, index=X_val_perm.columns)
    # perm_importances.append(perm_series)


print("CV summary")
print(f"XGB_BAL_ACC mean={np.mean(xgb_fold_bal_acc):.5f} +/- {np.std(xgb_fold_bal_acc):.5f}")
print(f"CAT_BAL_ACC mean={np.mean(cat_fold_bal_acc):.5f} +/- {np.std(cat_fold_bal_acc):.5f}")
print(f"LGBM_BAL_ACC mean={np.mean(lgbm_fold_bal_acc):.5f} +/- {np.std(lgbm_fold_bal_acc):.5f}")

# feature_importance_gain = pd.concat(gain_importances, axis=1).mean(axis=1).sort_values(ascending=False)
# feature_importance_perm = pd.concat(perm_importances, axis=1).mean(axis=1).sort_values(ascending=False)

# print("Top 10 (Gain):")
# print(feature_importance_gain.head(10))
# print("Top 10 (Permutation):")
# print(feature_importance_perm.head(10))


[0]	validation_0-mlogloss:1.08460	validation_0-bal_ACC:0.96930
[1000]	validation_0-mlogloss:0.06085	validation_0-bal_ACC:0.97224
[2000]	validation_0-mlogloss:0.05407	validation_0-bal_ACC:0.97262
[2061]	validation_0-mlogloss:0.05383	validation_0-bal_ACC:0.97269


d:\baobi\Irrigation\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [01:23:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Fold 1: XGB_BAL_ACC=0.97298, n_features=94
0:	learn: 0.9698609	test: 0.9680564	best: 0.9680564 (0)	total: 38.2ms	remaining: 6m 21s
1000:	learn: 0.9794266	test: 0.9724586	best: 0.9724682 (999)	total: 18.6s	remaining: 2m 47s
bestTest = 0.9727571835
bestIteration = 1478
Shrink model to first 1479 iterations.
Fold 1: CAT_BAL_ACC=0.97275, n_features=94
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.837008
[LightGBM] [Info] Total Bins 10095
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 94
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 2060 SUPER, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 16
[LightGBM] [Info] 90 dense feature groups (44.22 MB) transferred to GPU in 0.027402 secs. 1 sparse feature groups
[LightGBM] [Info] Start training 

Importance

In [13]:
# if "feature_importance_gain" not in globals() or "feature_importance_perm" not in globals():
#     print("Run the CV training cell first to compute feature_importance_gain / feature_importance_perm.")
# else:
#     TOPN = 25

#     plt.figure(figsize=(8, 10))
#     feature_importance_gain.head(TOPN).sort_values().plot(kind="barh")
#     plt.title("Top Feature Importance (XGBoost Gain, CV mean)")
#     plt.tight_layout()
#     plt.show()

#     plt.figure(figsize=(8, 10))
#     feature_importance_perm.head(TOPN).sort_values().plot(kind="barh")
#     plt.title("Top Feature Importance (Permutation, CV mean)")
#     plt.tight_layout()
#     plt.show()


Ensemble


In [ ]:
# Simple average
avg_oof = (xgb_oof_proba + cat_oof_proba + lgbm_oof_proba) / 3
avg_test = (xgb_test_proba_folds + cat_test_proba_folds + lgbm_test_proba_folds) / 3
avg_score = balanced_accuracy_score(y_encoded, avg_oof.argmax(1))
print(f'Simple average OOF: {avg_score:.5f}')

# LR stacking
oof_stack = np.hstack([xgb_oof_proba, cat_oof_proba, lgbm_oof_proba])
test_stack = np.hstack([xgb_test_proba_folds, cat_test_proba_folds, lgbm_test_proba_folds])

meta_oof = np.zeros(len(y_encoded), dtype=int)
meta_test = np.zeros((len(X_test), 3))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_encoded)):
    meta = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    meta.fit(oof_stack[tr_idx], y_encoded[tr_idx])
    meta_oof[val_idx] = meta.predict(oof_stack[val_idx])
    meta_test += meta.predict_proba(test_stack) / skf.n_splits

stack_score = balanced_accuracy_score(y_encoded, meta_oof)
print(f'LR stacking OOF: {stack_score:.5f}')

# Also try just XGB alone (reference notebook used single XGB)
xgb_alone = balanced_accuracy_score(y_encoded, xgb_oof_proba.argmax(1))
print(f'XGB alone OOF: {xgb_alone:.5f}')

# CAT alone
cat_alone = balanced_accuracy_score(y_encoded, cat_oof_proba.argmax(1))
print(f'CAT alone OOF: {cat_alone:.5f}')

# lgbm alone
lgbm_alone = balanced_accuracy_score(y_encoded, lgbm_oof_proba.argmax(1))
print(f'LGBM alone OOF: {lgbm_alone:.5f}')

# Pick best
scores = {
    'average': (avg_score, avg_test), 'stacking': (stack_score, None), 'xgb_alone': (xgb_alone, xgb_test_proba_folds), 
    'cat_alone': (cat_alone, cat_test_proba_folds), 'lgbm_alone': (lgbm_alone, lgbm_test_proba_folds)
}

best_name = max(scores, key=lambda k: scores[k][0])
print(f'\n>>> Best: {best_name} ({scores[best_name][0]:.5f})')

if best_name == 'stacking':
    final_meta = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    final_meta.fit(oof_stack, y_encoded)
    final_probs = final_meta.predict_proba(test_stack)
elif best_name == 'xgb_alone':
    final_probs = xgb_test_proba_folds
elif best_name == 'cat_alone':
    final_probs = cat_test_proba_folds
elif best_name == 'lgbm_alone':
    final_probs = lgbm_test_proba_folds
else:
    final_probs = avg_test

final_preds = final_probs.argmax(axis=1)
final_pred_label = pd.Series(final_preds).map(ID2LABEL).to_numpy()

submission = sample_sub.copy()

# find target column in sample submission (except id)
sub_target_cols = [c for c in submission.columns if c != "id"]
assert len(sub_target_cols) == 1, "sample_submission target column is not unique"
submission[sub_target_cols[0]] = final_pred_label

Simple average OOF: 0.97292
LR stacking OOF: 0.97456
XGB alone OOF: 0.97395
CAT alone OOF: 0.97356
LGBM alone OOF: 0.97033

>>> Best: stacking (0.97456)


Save

In [17]:
# save artifacts
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ARTIFACT_DIR = Path("results") / f"xgb_{RUN_ID}"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1) submission
submission_path = ARTIFACT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

# 2) model params + cv metrics
metrics = {
    "xgb_fold_bal_acc_mean": np.mean(xgb_fold_bal_acc),
    "xgb_fold_bal_acc_std": np.std(xgb_fold_bal_acc),
    "xgb_fold_bal_acc": xgb_fold_bal_acc,
    "cat_fold_bal_acc_mean": np.mean(cat_fold_bal_acc),
    "cat_fold_bal_acc_std": np.std(cat_fold_bal_acc),
    "cat_fold_bal_acc": cat_fold_bal_acc,
    "lgbm_fold_bal_acc_mean": np.mean(lgbm_fold_bal_acc),
    "lgbm_fold_bal_acc_std": np.std(lgbm_fold_bal_acc),
    "lgbm_fold_bal_acc": lgbm_fold_bal_acc,
    "avg_score": avg_score,
    "stack_score": stack_score,
    "xgb_alone": xgb_alone,
    "cat_alone": cat_alone,
    "lgbm_alone": lgbm_alone,
    "Best": best_name,
    "classes": [ID2LABEL[i] for i in sorted(ID2LABEL)],
    "n_features": len(train_columns),
}

serializable_model_params = {
    k: (v.__name__ if callable(v) else v)
    for k, v in xgb_params.items()
}

model_and_metrics = {
    "model_params": serializable_model_params,
    "cv_metrics": metrics,
}

model_metrics_path = ARTIFACT_DIR / "model_and_metrics.json"
with open(model_metrics_path, "w", encoding="utf-8") as f:
    json.dump(model_and_metrics, f, ensure_ascii=False, indent=2)

# 3) train columns
columns_path = ARTIFACT_DIR / "train_columns.txt"
with open(columns_path, "w", encoding="utf-8") as f:
    for c in train_columns:
        f.write(f"{c}\n")

print("saved:", submission_path)
print("saved:", model_metrics_path)
print("saved:", columns_path)




saved: results\xgb_20260409_014819\submission.csv
saved: results\xgb_20260409_014819\model_and_metrics.json
saved: results\xgb_20260409_014819\train_columns.txt
